# Chapter 11: Motion Models

<a href="../lite/lab/index.html?path=ch11_motion_models.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

Tell a robot "drive forward 1 meter" a hundred times. Plot where it ends up each time. Repeat this experiment 1000 times. You will get a banana shaped cloud of positions, not a neat circle. The shape of that banana tells you everything about how your motion model fails, and designing for it is the first step to building a reliable SLAM system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 11.1 Kinematics: Differential Drive

The simplest mobile robot has two wheels. Give it a **forward velocity** $v$ and a **turning rate** $\omega$, and it traces a circular arc:

$$\begin{aligned}
x_{t+1} &= x_t + v \cos(\theta_t)\, \Delta t \\
y_{t+1} &= y_t + v \sin(\theta_t)\, \Delta t \\
\theta_{t+1} &= \theta_t + \omega\, \Delta t
\end{aligned}$$

This is the **velocity model**: inputs are $(v, \omega)$ and the state is $(x, y, \theta)$.

**Try it:** Change the forward velocity and turning rate below.

```{admonition} What you will build
:class: tip

- Simulate differential drive, Ackermann, and omnidirectional robot kinematics
- Generate the banana shaped noise distribution from 500 noisy motion trials
- Watch IMU integration drift catastrophically over 60 seconds
- Implement the velocity motion model from Probabilistic Robotics and sample particles from it

**Real world application:** Every SLAM system needs a motion model for prediction. After this chapter, you will be able to simulate realistic robot motion with calibrated noise for testing your algorithms.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **nav_msgs/Odometry (ROS 2)** | Standard ROS message type for robot odometry |
| **robot_localization (ROS 2)** | EKF/UKF package that fuses odometry from multiple sources |
| **PyBullet** | Physics simulator for testing motion models with realistic dynamics |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
v      = 1.0    # forward velocity (m/s)   (try 0.5, 1.0, 2.0)
omega  = 0.1    # turning rate (rad/s)     (try 0.0, 0.1, 0.3, 0.5)
n_steps = 50    # number of time steps     (try 20, 50, 100)
dt     = 0.1    # time step (s)
# ─────────────────────────────────────────────────────────────────────────────

x, y, theta = 0.0, 0.0, 0.0
trajectory = [(x, y, theta)]

for _ in range(n_steps):
    x += v * np.cos(theta) * dt
    y += v * np.sin(theta) * dt
    theta += omega * dt
    trajectory.append((x, y, theta))

traj = np.array(trajectory)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(traj[:, 0], traj[:, 1], 'steelblue', lw=2.5, label='Trajectory')
ax.plot(traj[0, 0], traj[0, 1], 'o', color='forestgreen', ms=10, label='Start', zorder=5)
ax.plot(traj[-1, 0], traj[-1, 1], 's', color='tomato', ms=10, label='End', zorder=5)

# Draw heading arrows every few steps
arrow_step = max(1, n_steps // 8)
for i in range(0, len(traj), arrow_step):
    dx = 0.15 * np.cos(traj[i, 2])
    dy = 0.15 * np.sin(traj[i, 2])
    ax.arrow(traj[i, 0], traj[i, 1], dx, dy, head_width=0.06,
             head_length=0.03, fc='tomato', ec='tomato')

ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Differential Drive: v={v}, \u03c9={omega}')
ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.show()

if abs(omega) > 1e-6:
    R = v / omega
    print(f'Turning radius: {R:.2f} m')
else:
    print('Driving straight (omega \u2248 0)')

**Key observations:**
- When $\omega = 0$, the robot drives in a straight line.
- When $\omega > 0$, the robot turns left along a circular arc with radius $R = v / \omega$.
- Larger $\omega$ means tighter turns.

## 11.2 Vehicle Models

Not all robots move the same way. Three common kinematic models:

1. **Differential drive** (most mobile robots): two wheels, independent speeds. The robot can turn in place.
2. **Ackermann steering** (cars): front wheels steer, rear wheels are fixed. The robot cannot turn in place; it has a minimum turning radius.
3. **Omnidirectional** (Mecanum wheels): the robot can move in any direction without rotating.

The Ackermann model adds a **steering angle** $\delta$ and a **wheelbase** $L$:

$$\begin{aligned}
x_{t+1} &= x_t + v \cos(\theta_t)\, \Delta t \\
y_{t+1} &= y_t + v \sin(\theta_t)\, \Delta t \\
\theta_{t+1} &= \theta_t + \frac{v}{L} \tan(\delta)\, \Delta t
\end{aligned}$$

The omnidirectional model decouples translation from rotation:

$$\begin{aligned}
x_{t+1} &= x_t + v_x \cos(\theta_t) \Delta t - v_y \sin(\theta_t) \Delta t \\
y_{t+1} &= y_t + v_x \sin(\theta_t) \Delta t + v_y \cos(\theta_t) \Delta t \\
\theta_{t+1} &= \theta_t + \omega \Delta t
\end{aligned}$$

**Try it:** Compare all three models with similar velocity commands.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
v_cmd      = 1.0    # forward speed (m/s)       (try 0.5, 1.0, 2.0)
omega_cmd  = 0.15   # turning rate (rad/s)      (try 0.05, 0.15, 0.4)
n_steps    = 80     # number of steps           (try 40, 80, 150)
dt         = 0.1    # time step
wheelbase  = 0.5    # Ackermann wheelbase (m)   (try 0.3, 0.5, 1.0)
# ─────────────────────────────────────────────────────────────────────────────

def simulate_diff_drive(v, omega, n, dt):
    x, y, th = 0.0, 0.0, 0.0
    path = [(x, y)]
    for _ in range(n):
        x += v * np.cos(th) * dt
        y += v * np.sin(th) * dt
        th += omega * dt
        path.append((x, y))
    return np.array(path)

def simulate_ackermann(v, omega, n, dt, L):
    # Convert omega to steering angle: omega = v/L * tan(delta)
    if abs(v) > 1e-6:
        delta = np.arctan(omega * L / v)
    else:
        delta = 0.0
    x, y, th = 0.0, 0.0, 0.0
    path = [(x, y)]
    for _ in range(n):
        x += v * np.cos(th) * dt
        y += v * np.sin(th) * dt
        th += (v / L) * np.tan(delta) * dt
        path.append((x, y))
    return np.array(path)

def simulate_omni(v, omega, n, dt):
    # Omnidirectional: same forward speed, but also a lateral component
    vx, vy = v, 0.3 * v  # slight lateral drift to show capability
    x, y, th = 0.0, 0.0, 0.0
    path = [(x, y)]
    for _ in range(n):
        x += (vx * np.cos(th) - vy * np.sin(th)) * dt
        y += (vx * np.sin(th) + vy * np.cos(th)) * dt
        th += omega * dt
        path.append((x, y))
    return np.array(path)

path_dd  = simulate_diff_drive(v_cmd, omega_cmd, n_steps, dt)
path_ack = simulate_ackermann(v_cmd, omega_cmd, n_steps, dt, wheelbase)
path_omn = simulate_omni(v_cmd, omega_cmd, n_steps, dt)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ['Differential Drive', f'Ackermann (L={wheelbase}m)', 'Omnidirectional']
colors = ['steelblue', 'tomato', 'forestgreen']
paths  = [path_dd, path_ack, path_omn]

for ax, path, title, color in zip(axes, paths, titles, colors):
    ax.plot(path[:, 0], path[:, 1], color=color, lw=2.5)
    ax.plot(path[0, 0], path[0, 1], 'o', color='forestgreen', ms=8, zorder=5)
    ax.plot(path[-1, 0], path[-1, 1], 's', color='tomato', ms=8, zorder=5)
    ax.set_title(title); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_aspect('equal')

plt.suptitle(f'Three Vehicle Models: v={v_cmd}, \u03c9={omega_cmd}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

**Things to notice:**
- The differential drive and Ackermann models produce similar arcs for the same $(v, \omega)$ pair.
- The omnidirectional robot can move sideways while turning, producing a spiral.
- The Ackermann model is constrained: it cannot turn in place ($v=0, \omega \neq 0$).

## 11.3 Process Noise: The Banana of Uncertainty

Real robots do not execute commands perfectly. **Process noise** enters through:
- **Wheel slip** on smooth or uneven surfaces
- **Motor imprecision** in commanded vs actual velocities
- **Terrain variation** that changes effective wheel radius

We model this by adding Gaussian noise to the velocity commands:

$$\hat{v} = v + \epsilon_v, \quad \epsilon_v \sim \mathcal{N}(0, \sigma_v^2)$$
$$\hat{\omega} = \omega + \epsilon_\omega, \quad \epsilon_\omega \sim \mathcal{N}(0, \sigma_\omega^2)$$

The resulting distribution of final positions is **not circular**. Because heading errors compound with forward motion, the cloud of endpoints forms a characteristic **banana shape** (a crescent aligned with the arc of travel).

**Try it:** Change the noise levels and the number of trials.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
sigma_v     = 0.05   # velocity noise std (m/s)    (try 0.01, 0.05, 0.15)
sigma_omega = 0.02   # turning rate noise (rad/s)  (try 0.005, 0.02, 0.08)
n_trials    = 500    # number of Monte Carlo runs  (try 100, 500, 2000)
v_nom       = 1.0    # nominal velocity
omega_nom   = 0.1    # nominal turning rate
n_steps     = 50     # steps per trial
dt          = 0.1    # time step
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
endpoints = np.zeros((n_trials, 3))  # x, y, theta

for trial in range(n_trials):
    x, y, theta = 0.0, 0.0, 0.0
    for _ in range(n_steps):
        v_noisy = v_nom + np.random.normal(0, sigma_v)
        w_noisy = omega_nom + np.random.normal(0, sigma_omega)
        x += v_noisy * np.cos(theta) * dt
        y += v_noisy * np.sin(theta) * dt
        theta += w_noisy * dt
    endpoints[trial] = [x, y, theta]

# Noiseless reference
x0, y0, th0 = 0.0, 0.0, 0.0
ref_path = [(x0, y0)]
for _ in range(n_steps):
    x0 += v_nom * np.cos(th0) * dt
    y0 += v_nom * np.sin(th0) * dt
    th0 += omega_nom * dt
    ref_path.append((x0, y0))
ref_path = np.array(ref_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(endpoints[:, 0], endpoints[:, 1], s=5, alpha=0.4, color='steelblue', label='Endpoints')
ax.plot(ref_path[:, 0], ref_path[:, 1], 'k--', lw=1.5, label='Noiseless path')
ax.plot(ref_path[-1, 0], ref_path[-1, 1], 'x', color='tomato', ms=12, mew=3, label='Ideal endpoint')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Final Positions ({n_trials} trials)\n\u03c3_v={sigma_v}, \u03c3_\u03c9={sigma_omega}')
ax.set_aspect('equal'); ax.legend(fontsize=9)

ax2 = axes[1]
ax2.scatter(endpoints[:, 0], endpoints[:, 1], s=5, alpha=0.3, color='steelblue')
# Compute and plot covariance ellipse
mean_xy = endpoints[:, :2].mean(axis=0)
cov_xy = np.cov(endpoints[:, :2].T)
evals, evecs = np.linalg.eigh(cov_xy)
angle = np.degrees(np.arctan2(evecs[1, 1], evecs[0, 1]))
import matplotlib.patches as mpatches
for ns, color, lbl in [(1, 'tomato', '1\u03c3'), (2, 'orange', '2\u03c3'), (3, 'forestgreen', '3\u03c3')]:
    ell = mpatches.Ellipse(mean_xy, 2*ns*np.sqrt(evals[1]), 2*ns*np.sqrt(evals[0]),
                           angle=angle, fill=False, edgecolor=color, linewidth=2, label=lbl)
    ax2.add_patch(ell)
ax2.set_xlabel('x (m)'); ax2.set_ylabel('y (m)')
ax2.set_title('The "Banana" Shape + Covariance Ellipse')
ax2.set_aspect('equal'); ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Mean endpoint:   ({mean_xy[0]:.3f}, {mean_xy[1]:.3f})')
print(f'Std dev x:       {np.sqrt(cov_xy[0,0]):.4f} m')
print(f'Std dev y:       {np.sqrt(cov_xy[1,1]):.4f} m')
print(f'Correlation:     {cov_xy[0,1]/np.sqrt(cov_xy[0,0]*cov_xy[1,1]):.3f}')

**Why a banana?** When $\sigma_\omega$ causes the heading to drift, the robot arcs off course. The forward velocity noise stretches endpoints along the direction of travel, while the heading noise bends them into a crescent. The covariance ellipse captures the overall spread, but notice the shape is not perfectly Gaussian; it curves.

## 11.4 Constant Velocity and Constant Acceleration Models

For prediction in tracking systems, we often use simple **linear** motion models:

**Constant velocity (CV):**
$$\mathbf{x}_{t+1} = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix} \mathbf{x}_t + \mathbf{w}_t, \qquad \mathbf{x} = \begin{bmatrix} p \\ \dot{p} \end{bmatrix}$$

**Constant acceleration (CA):**
$$\mathbf{x}_{t+1} = \begin{bmatrix} 1 & \Delta t & \tfrac{1}{2}\Delta t^2 \\ 0 & 1 & \Delta t \\ 0 & 0 & 1 \end{bmatrix} \mathbf{x}_t + \mathbf{w}_t, \qquad \mathbf{x} = \begin{bmatrix} p \\ \dot{p} \\ \ddot{p} \end{bmatrix}$$

These are **state transition matrices** $F$. The process noise $\mathbf{w}_t \sim \mathcal{N}(0, Q)$ models unmodeled accelerations or jerk.

**Try it:** Compare the constant velocity prediction against an actual accelerating target.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
true_accel   = 0.3    # actual acceleration of the target  (try 0, 0.3, 1.0)
process_std  = 0.2    # process noise std dev              (try 0.05, 0.2, 0.5)
n_steps      = 60     # number of time steps               (try 30, 60, 100)
dt           = 0.1    # time step
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(7)

# Ground truth: constant acceleration
t = np.arange(n_steps) * dt
true_pos = 0.5 * true_accel * t**2 + 1.0 * t  # p = 0.5*a*t^2 + v0*t
true_vel = true_accel * t + 1.0

# Constant velocity model prediction (no updates, just open loop)
F_cv = np.array([[1, dt], [0, 1]])
Q_cv = process_std**2 * np.array([[dt**3/3, dt**2/2], [dt**2/2, dt]])

state = np.array([0.0, 1.0])  # initial [position, velocity]
pred_pos = [state[0]]
pred_vel = [state[1]]

# With noise
state_noisy = np.array([0.0, 1.0])
noisy_pos = [state_noisy[0]]

for i in range(n_steps - 1):
    state = F_cv @ state
    pred_pos.append(state[0])
    pred_vel.append(state[1])
    
    noise = np.random.multivariate_normal([0, 0], Q_cv)
    state_noisy = F_cv @ state_noisy + noise
    noisy_pos.append(state_noisy[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(t, true_pos, 'k-', lw=2.5, label='True (accelerating)')
ax1.plot(t, pred_pos, 'steelblue', lw=2, ls='--', label='CV prediction (noiseless)')
ax1.plot(t, noisy_pos, 'tomato', lw=1.5, alpha=0.8, label='CV prediction + noise')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Position (m)')
ax1.set_title('Constant Velocity Model vs Accelerating Target')
ax1.legend(fontsize=9)

# Show prediction error
error = np.array(pred_pos) - true_pos
ax2.plot(t, error, 'tomato', lw=2)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Position Error (m)')
ax2.set_title('Prediction Error Grows Quadratically')
ax2.fill_between(t, error, alpha=0.15, color='tomato')

plt.tight_layout(); plt.show()
print(f'Final prediction error: {error[-1]:.2f} m')
print(f'The CV model assumes constant velocity, so it underestimates')
print(f'an accelerating target by ~0.5 * a * T\u00b2 = {0.5*true_accel*(t[-1])**2:.2f} m')

**Takeaway:** The constant velocity model works well when the target is not accelerating. When it is, the prediction error grows quadratically. A constant acceleration model would handle this better, at the cost of estimating one more state variable.

## 11.5 IMU Connection: Integration Drift

An **Inertial Measurement Unit** (IMU) provides:
- **Accelerometer:** measures linear acceleration $a$
- **Gyroscope:** measures angular velocity $\omega$

To get position from an accelerometer, we must integrate twice:

$$v(t) = \int_0^t a(\tau)\, d\tau, \qquad p(t) = \int_0^t v(\tau)\, d\tau$$

Each integration amplifies noise. A tiny bias in the accelerometer becomes a linearly growing velocity error, which becomes a **quadratically growing** position error. This is the fundamental problem of **dead reckoning** with IMUs.

**Try it:** Change the sensor noise and bias.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
accel_noise_std = 0.1    # accelerometer noise std (m/s\u00b2)  (try 0.01, 0.1, 0.5)
accel_bias      = 0.02   # constant accelerometer bias     (try 0.0, 0.02, 0.1)
duration        = 10.0   # simulation duration (s)         (try 5, 10, 20)
dt              = 0.01   # sample rate (s)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(12)
n = int(duration / dt)
t = np.arange(n) * dt

# True motion: constant velocity (zero acceleration)
true_accel = np.zeros(n)
true_vel = np.ones(n) * 1.0   # 1 m/s constant
true_pos = true_vel[0] * t

# Noisy IMU readings
measured_accel = true_accel + accel_bias + np.random.normal(0, accel_noise_std, n)

# Integrate to get velocity and position
est_vel = np.zeros(n)
est_pos = np.zeros(n)
est_vel[0] = 1.0  # assume we know initial velocity
est_pos[0] = 0.0

for i in range(1, n):
    est_vel[i] = est_vel[i-1] + measured_accel[i] * dt
    est_pos[i] = est_pos[i-1] + est_vel[i] * dt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(t, measured_accel, 'steelblue', alpha=0.4, lw=0.5)
axes[0].axhline(0, color='k', lw=1, label='True acceleration')
axes[0].axhline(accel_bias, color='tomato', ls='--', label=f'Bias = {accel_bias}')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Acceleration (m/s\u00b2)')
axes[0].set_title('IMU Acceleration Readings'); axes[0].legend(fontsize=8)

axes[1].plot(t, true_vel, 'k-', lw=2, label='True velocity')
axes[1].plot(t, est_vel, 'tomato', lw=1.5, label='Integrated velocity')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Velocity (m/s)')
axes[1].set_title('Velocity Drift (1st integral)'); axes[1].legend(fontsize=8)

axes[2].plot(t, true_pos, 'k-', lw=2, label='True position')
axes[2].plot(t, est_pos, 'tomato', lw=1.5, label='Integrated position')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Position (m)')
axes[2].set_title('Position Drift (2nd integral)'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

pos_error = est_pos[-1] - true_pos[-1]
print(f'Position error after {duration}s: {pos_error:.2f} m')
print(f'Expected bias induced drift: 0.5 * bias * T\u00b2 = {0.5 * accel_bias * duration**2:.2f} m')

**The triple integration curse:** Gyroscope noise drifts into heading error; accelerometer noise drifts into velocity error; velocity error drifts into position error. After just a few seconds, pure IMU dead reckoning is wildly off. This is why SLAM systems fuse IMU data with other sensors (cameras, LiDAR) to correct the drift.

## 11.6 Drift: Quantifying Uncertainty Growth

How fast does uncertainty grow with dead reckoning? For a simple random walk model:

- **Heading uncertainty** grows as $\sigma_\theta \propto \sqrt{t}$ (random walk in angle)
- **Position uncertainty** grows linearly in $t$ because heading errors couple into position

More precisely, for small heading noise, the position standard deviation grows roughly as:

$$\sigma_x(t) \approx \sigma_v \sqrt{t\, \Delta t} + v \sigma_\omega t \sqrt{\frac{t\, \Delta t}{3}}$$

The first term is from velocity noise (grows as $\sqrt{t}$). The second term is from heading noise (grows faster, roughly as $t^{3/2}$).

**Try it:** See how different noise levels affect drift growth.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
noise_levels = [0.01, 0.03, 0.05, 0.10]  # sigma_omega values to compare
sigma_v      = 0.05    # velocity noise (fixed)   (try 0.01, 0.05, 0.1)
v_nom        = 1.0     # nominal velocity
n_steps_max  = 200     # max steps to simulate
n_trials     = 300     # Monte Carlo trials per noise level
dt           = 0.1     # time step
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
step_range = np.arange(1, n_steps_max + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors_list = ['steelblue', 'tomato', 'orange', 'forestgreen']

for idx, sigma_w in enumerate(noise_levels):
    pos_errors = np.zeros((n_trials, n_steps_max))
    heading_errors = np.zeros((n_trials, n_steps_max))
    
    for trial in range(n_trials):
        x, y, theta = 0.0, 0.0, 0.0
        x_true, y_true, th_true = 0.0, 0.0, 0.0
        for step in range(n_steps_max):
            # True trajectory (noiseless, going straight)
            x_true += v_nom * dt
            
            # Noisy trajectory
            v_n = v_nom + np.random.normal(0, sigma_v)
            w_n = np.random.normal(0, sigma_w)
            x += v_n * np.cos(theta) * dt
            y += v_n * np.sin(theta) * dt
            theta += w_n * dt
            
            pos_errors[trial, step] = np.sqrt((x - x_true)**2 + (y - y_true)**2)
            heading_errors[trial, step] = abs(theta)
    
    pos_std = pos_errors.std(axis=0)
    head_std = heading_errors.std(axis=0)
    c = colors_list[idx % len(colors_list)]
    ax1.plot(step_range, pos_std, color=c, lw=2, label=f'\u03c3_\u03c9 = {sigma_w}')
    ax2.plot(step_range, head_std, color=c, lw=2, label=f'\u03c3_\u03c9 = {sigma_w}')

ax1.set_xlabel('Steps'); ax1.set_ylabel('Position Std Dev (m)')
ax1.set_title('Position Error Growth'); ax1.legend(fontsize=9)
ax2.set_xlabel('Steps'); ax2.set_ylabel('Heading Std Dev (rad)')
ax2.set_title('Heading Error Growth'); ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()
print('Heading error grows as \u221at (random walk).')
print('Position error grows faster because heading errors steer the robot off course.')

## 11.7 Capstone: Velocity Motion Model with Particle Sampling

The **velocity motion model** from Thrun (Probabilistic Robotics, Chapter 5) samples from:

$$p(x_t \mid x_{t-1}, u_t)$$

where $u_t = (v, \omega)$. We add noise proportional to the command magnitudes:

$$\hat{v} = v + \epsilon_1, \quad \hat{\omega} = \omega + \epsilon_2, \quad \hat{\gamma} = \epsilon_3$$

where $\epsilon_1, \epsilon_2, \epsilon_3$ are zero mean Gaussians with variances that depend on $\alpha_1 \ldots \alpha_4$ (noise parameters).

We will simulate a robot driving a **figure 8** pattern, then sample 300 particles at each step to visualize how the noise distribution evolves.

**Try it:** Adjust the noise parameters to see how the particle cloud changes.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
alpha1 = 0.005   # velocity  -> velocity noise  (try 0.001, 0.005, 0.02)
alpha2 = 0.005   # omega     -> velocity noise  (try 0.001, 0.005, 0.02)
alpha3 = 0.01    # velocity  -> omega noise     (try 0.005, 0.01, 0.05)
alpha4 = 0.01    # omega     -> omega noise     (try 0.005, 0.01, 0.05)
alpha5 = 0.002   # additional heading drift     (try 0.001, 0.002, 0.01)
alpha6 = 0.002   # additional heading drift     (try 0.001, 0.002, 0.01)
n_particles = 300  # particles per time step    (try 100, 300, 1000)
# ─────────────────────────────────────────────────────────────────────────────

def velocity_motion_model_sample(x, y, theta, v, omega, dt, alphas):
    """Sample from the velocity motion model (Thrun Ch5)."""
    a1, a2, a3, a4, a5, a6 = alphas
    
    # Add noise proportional to command magnitudes
    v_hat = v + np.random.normal(0, np.sqrt(a1 * v**2 + a2 * omega**2))
    w_hat = omega + np.random.normal(0, np.sqrt(a3 * v**2 + a4 * omega**2))
    gamma = np.random.normal(0, np.sqrt(a5 * v**2 + a6 * omega**2))
    
    # Apply motion
    if abs(w_hat) > 1e-6:
        r = v_hat / w_hat
        x_new = x - r * np.sin(theta) + r * np.sin(theta + w_hat * dt)
        y_new = y + r * np.cos(theta) - r * np.cos(theta + w_hat * dt)
    else:
        x_new = x + v_hat * np.cos(theta) * dt
        y_new = y + v_hat * np.sin(theta) * dt
    theta_new = theta + w_hat * dt + gamma * dt
    
    return x_new, y_new, theta_new

# Generate figure 8 commands
dt = 0.1
total_time = 12.0
n_cmd = int(total_time / dt)
t_arr = np.arange(n_cmd) * dt

# Figure 8: sinusoidal omega
v_cmds = np.ones(n_cmd) * 1.0
omega_cmds = 0.8 * np.sin(2 * np.pi * t_arr / (total_time / 2))

alphas = (alpha1, alpha2, alpha3, alpha4, alpha5, alpha6)

# Compute noiseless trajectory
x_ref, y_ref, th_ref = 0.0, 0.0, 0.0
ref_traj = [(x_ref, y_ref)]
for i in range(n_cmd):
    v, w = v_cmds[i], omega_cmds[i]
    if abs(w) > 1e-6:
        r = v / w
        x_ref = x_ref - r * np.sin(th_ref) + r * np.sin(th_ref + w * dt)
        y_ref = y_ref + r * np.cos(th_ref) - r * np.cos(th_ref + w * dt)
    else:
        x_ref += v * np.cos(th_ref) * dt
        y_ref += v * np.sin(th_ref) * dt
    th_ref += w * dt
    ref_traj.append((x_ref, y_ref))
ref_traj = np.array(ref_traj)

# Sample particles at selected time steps
np.random.seed(42)
snapshot_steps = np.linspace(0, n_cmd - 1, 10, dtype=int)

fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(ref_traj[:, 0], ref_traj[:, 1], 'k-', lw=2, label='Noiseless path', zorder=3)

for snap_idx in snapshot_steps:
    particles_x = []
    particles_y = []
    for _ in range(n_particles):
        px, py, pth = 0.0, 0.0, 0.0
        for i in range(snap_idx + 1):
            px, py, pth = velocity_motion_model_sample(
                px, py, pth, v_cmds[i], omega_cmds[i], dt, alphas)
        particles_x.append(px)
        particles_y.append(py)
    ax.scatter(particles_x, particles_y, s=3, alpha=0.3, color='steelblue', zorder=2)

# Mark snapshot positions on reference
for snap_idx in snapshot_steps:
    ax.plot(ref_traj[snap_idx + 1, 0], ref_traj[snap_idx + 1, 1],
            'o', color='tomato', ms=6, zorder=4)

ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Velocity Motion Model: Particle Clouds Along Figure 8\n'
             f'({n_particles} particles per snapshot, \u03b1\u2081={alpha1}, \u03b1\u2083={alpha3})')
ax.set_aspect('equal')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

**What to observe:**
- The particle cloud grows over time as noise accumulates.
- The cloud is not circular; it stretches along the direction of motion (the banana effect).
- Larger $\alpha$ values produce wider spreads. The $\alpha_3, \alpha_4$ parameters (heading noise) have a stronger effect than $\alpha_1, \alpha_2$ (speed noise) because heading errors compound.
- This is exactly the sampling step used inside a **particle filter**, which we will study in a later chapter.

## Exercises

**Exercise 11.1:** Implement the **odometry motion model** (Thrun Ch5.4). Instead of $(v, \omega)$, the input is a pair of relative poses: $(\bar{\delta}_{rot1}, \bar{\delta}_{trans}, \bar{\delta}_{rot2})$. Sample 500 particles for a command of $(\delta_{rot1}=30\degree, \delta_{trans}=1\text{m}, \delta_{rot2}=15\degree)$ and plot them.

In [ ]:
# Hint: The odometry model decomposes motion into rotate, translate, rotate.
# Add noise to each component proportional to the command magnitude.
# Your code here

**Exercise 11.2:** For the differential drive model, derive the **Jacobian** $G = \frac{\partial g}{\partial \mathbf{x}}$ where $g$ is the motion function. Compute $G$ numerically for $(v=1, \omega=0.1, \theta=0.5)$ and verify it against an analytical expression.

In [ ]:
# Hint: The Jacobian of the diff drive model with respect to [x, y, theta] is:
# G = [[1, 0, -v*sin(theta)*dt],
#       [0, 1,  v*cos(theta)*dt],
#       [0, 0,  1              ]]
# Compare this to a finite difference approximation.
# Your code here

**Exercise 11.3:** Compare **dead reckoning drift** for a differential drive vs an Ackermann vehicle. Drive both models in a straight line with the same noise parameters for 500 steps. Which model accumulates more lateral error, and why?

In [ ]:
# Hint: Run Monte Carlo simulations (e.g., 200 trials)
# for each model and compare the spread of final positions.
# Your code here

**Exercise 11.4 (challenge):** Implement a **constant acceleration** motion model in 2D. The state is $[x, y, v_x, v_y, a_x, a_y]$. Construct the transition matrix $F$ and process noise $Q$. Simulate a target that accelerates for the first half and decelerates for the second half. Compare the prediction quality of a constant velocity model vs a constant acceleration model.

In [ ]:
# Hint: The 2D constant acceleration transition matrix is 6x6.
# F = [[1, 0, dt, 0, 0.5*dt^2, 0       ],
#       [0, 1, 0,  dt, 0,       0.5*dt^2],
#       [0, 0, 1,  0,  dt,      0       ],
#       [0, 0, 0,  1,  0,       dt      ],
#       [0, 0, 0,  0,  1,       0       ],
#       [0, 0, 0,  0,  0,       1       ]]
# Your code here